Chroma stores documents in collections, lets you add text with unique IDs, and query similar chunks later. OpenRouter uses an OpenAI-compatible /api/v1/chat/completions endpoint.

##1. Install

In [1]:
%%capture
pip install chromadb openai

## 2. Loading api key

In [2]:
from google.colab import userdata
api_key=userdata.get('main_api')

In [3]:
import os
import chromadb
from openai import OpenAI
from dotenv import load_dotenv

In [4]:
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
)

Chroma is the open-source data infrastructure for AI. It comes with everything you need to get started built-in.

Documentation Index
Fetch the complete documentation index at: https://docs.trychroma.com/llms.txt

Use this file to discover all available pages before exploring further.

Chroma gives you everything you need for retrieval: store embeddings with metadata, search with dense and sparse vectors, filter by metadata, and retrieve across text, images, and more.

In [6]:
docs = [
    """
    Document ID: school_student_001
    Category: Student Profiles
    Student Name: Leo Miller (Grade 3)

    Leo Miller is a third-grade student in Ms. Davis's class. His favorite hobby is
    building miniature wooden rocket ships, and he has a pet chameleon named 'Sparky'
    that turns bright blue when it sits on his backpack. Leo's library goal this term
    is to read 15 books about deep-sea creatures.
    """,
    """
    Document ID: school_policy_002
    Category: School Rules
    Subject: The Friday Playground 'Golden Rule'

    At Oakwood Elementary, Fridays have a unique playground rule called the 'Golden Ball Protocol'.
    If a student finds the hidden yellow kickball under the oak tree before morning recess,
    their entire classroom gets an extra 10 minutes of free time during lunch. The ball
    must be returned directly to Principal green, and no running is allowed while holding it.
    """,
    """
    Document ID: school_event_003
    Category: Field Trips
    Subject: 2026 Annual Spring Science Trip

    The 2026 Grade 3 field trip is scheduled for June 12th. Students will be visiting the
    'Eco-Dome Bio-Reserve'. Classrooms will travel by bus number 14. All students must bring
    a lunch packed entirely in reusable containers (zero-waste policy) and wear their bright
    purple school t-shirts for easy tracking by the chaperones.
    """,
    """
    Document ID: school_talent_004
    Category: Talent Show Records
    Subject: Grade 4 Talent Show Winners

    The winner of the Oakwood Elementary 2026 Talent Show was Maya Lin from Grade 4.
    Maya won first place by playing the theme song of her favorite cartoon using only
    musical water glasses filled to different levels. Second place went to a group
    juggling act using neon green tennis balls.
    """
]

ids = ["school_student_001", "school_policy_002", "school_event_003", "school_talent_004"]

In [7]:
chroma_client = chromadb.PersistentClient(path="./chroma_db")
# Delete the collection if it already exists to ensure a clean slate
try:
    chroma_client.delete_collection(name="ai_agent_notes")
except:
    pass # Collection might not exist yet
collection = chroma_client.get_or_create_collection(
    name="ai_agent_notes"
)

In [8]:
collection.add(
    documents=docs,
    ids=ids
)


## There’s a lot left to consider, but the core building blocks are here. Some next steps to consider:
  1. **Embedding Model** There are many embedding models on the market, some optimized for code, others for english and others still for various languages. Embedding model selection plays a big role in retrieval accuracy.
 2. **Chunking Chunking** strategies are very unique to the data. Deciding how large or small to make chunks is critical to the performance of the system.
 3. **n_results** varying the number of results balances token usage with correctness. The more results, the likely the better answer from the LLM but at the expense of more token usage.

In [13]:
question = "What color does Leo Miller's pet chameleon turn when sitting on his backpack?"
results = collection.query(
    query_texts=[question],
    n_results=1
)
print(results)

{'ids': [['school_student_001']], 'embeddings': None, 'documents': [["\n    Document ID: school_student_001\n    Category: Student Profiles\n    Student Name: Leo Miller (Grade 3)\n    \n    Leo Miller is a third-grade student in Ms. Davis's class. His favorite hobby is \n    building miniature wooden rocket ships, and he has a pet chameleon named 'Sparky' \n    that turns bright blue when it sits on his backpack. Leo's library goal this term \n    is to read 15 books about deep-sea creatures.\n    "]], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[None]], 'distances': [[0.8793508410453796]]}


In [15]:
retrieved_docs = results["documents"][0]

context = "\n\n".join(retrieved_docs)

In [19]:
# 5. Send context to Qwen
# -----------------------------
system_prompt = """
You are a helpful AI agent teacher.
Answer the user's question using only the provided context.
If the answer is not in the context, say you don't know.
"""

user_prompt = f"""

Context:
{context}
Question:
{question}
"""



In [20]:
response = client.chat.completions.create(
    model="qwen/qwen3.6-flash",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ],
    max_tokens=500
)

print(response.choices[0].message.content)

Based on the provided context, Leo Miller's pet chameleon turns bright blue when it sits on his backpack.


#### **What is happening? **
 * User question
 *    ↓
 * Search Chroma
  *  ↓
 * Get relevant documents
 *    ↓
 * Put documents into prompt
 *    ↓
 * Qwen answers using that context

**Better version with chunking**

In real projects, you should not store big files as one document. Split them into chunks.*

---



In [78]:
def chunk_text(text, chunk_size=800, overlap=300):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap

    return chunks

In [79]:
chroma_client = chromadb.PersistentClient(path="./chroma_db")
# Delete the collection if it already exists to ensure a clean slate
try:
    chroma_client.delete_collection(name="ai_agent_notes")
except:
    pass # Collection might not exist yet
collection = chroma_client.get_or_create_collection(
    name="ai_agent_notes"
)

In [80]:
big_text = """
Table of Contents
UNIT 1: LIVING IN RURAL AREA.1
UNIT 2: HISTORY OF CALENDAR.17
UNIT 3: ROAD SAFETY..30
UNIT 4: ENDEMIC ANIMALS IN ETHIOPIA..45
UNIT 5: DIARY..59
UNIT 6: LAND CONSERVATION.73
UNIT 7: VOLUNTEERISM...89
UNIT 8: FITNESS..105
UNIT 9: SELF-EXPRESSIONS..120
UNIT 10: COMPUTER DEVELOPMENT.134
SECTION THREE: READING
Lesson 5
Pre-reading
Activity 1.12: Answer the following questions before you read the passage. Use the picture
as a guide
1.What does life in rural and urban area look like?
2. What do you think the next passage will be about?
Read the following passage and answer the questions that follow.
Living in Rural and Urban Areas: Advantages and Disadvantages
There are a number of advantages and disadvantages of living in rural and urban areas. In rural
area, the air is fresh and healthy. There are fewer automobiles in villages, and there is an absence
of significant industrial sectors. Air pollution is lower, and there is green environment. On the
other hand, in urban areas or cities, with the advancement of technology, the environment is polluted
with gases from vehicles. Factories discharge water without treatment and emit poisonous into the
atmosphere; as a result, there is lack of pure water and fresh air. This pollution often makes urban
life suffocating and congested, so a vast number of urban people suffer from health problems such as
heart disease or breathing problems.
People living in cities go to villages during their holidays to take a break from the polluted and
contaminated urban environment. Villages have healthy and pleasant weather. The food in rural
area is natural and healthier than in cities where people mix different things.

Although people in urban areas suffer from different diseases as compared to rural area, there is
access to medical care. Moreover, compared to rural areas, access to medical treatment is easier
because there are always clinics or medical centers that open 24 hours in many parts of the city
to get medical help anytime. Public transport is also usually available 24 hours a day. Therefore,
living in big cities is more convenient because of the complete facilities such as transport,
education, health care and entertainment are provided.

"""

chunks=chunk_text(big_text)


In [81]:
collection.add(
    documents=chunks,
    ids=[f"chunk_{i}" for i in range(len(chunks))]
)

**Important beginner mistake**

**Do not do this:**
  - 1. Put all knowledge directly inside the prompt.

**Better:**
1. Store knowledge in Chroma
2. Retrieve only the relevant chunks
3. Send those chunks to the model

In [92]:
question = """Activity 1.14: Choose the best answer based on the information in the above passage.
1. Why do people who live in urban areas suffer from breathing problems?
A. Because they are busy working different activities.
B. Because of a large number of vehicles, the area is polluted.
C. Because they engage in a large number of social activities than rural people.
D. Because they prefer comfortable life.
"""
results = collection.query(
    query_texts=[question],
    n_results=3
)

In [93]:
retrieved_docs = results["documents"][0]
context = "\n\n".join(retrieved_docs)

In [94]:
# 5. Send context to Qwen
# -----------------------------
system_prompt = """
You are a helpful AI agent teacher.
Answer the user's question using only the provided context.
If the answer is not in the context, say you don't know.
"""

user_prompt = f"""

context:
{context}
Question:
{question}
"""


In [95]:
context

'as or cities, with the advancement of technology, the environment is polluted\nwith gases from vehicles. Factories discharge water without treatment and emit poisonous into the\natmosphere; as a result, there is lack of pure water and fresh air. This pollution often makes urban\nlife suffocating and congested, so a vast number of urban people suffer from health problems such as\nheart disease or breathing problems.\nPeople living in cities go to villages during their holidays to take a break from the polluted and\ncontaminated urban environment. Villages have healthy and pleasant weather. The food in rural\narea is natural and healthier than in cities where people mix different things.\n\nAlthough people in urban areas suffer from different diseases as compared to rural area, there is\naccess to med\n\n polluted and\ncontaminated urban environment. Villages have healthy and pleasant weather. The food in rural\narea is natural and healthier than in cities where people mix different thi

In [96]:
response = client.chat.completions.create(
    model="qwen/qwen3.6-flash",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ],
    max_tokens=500
)

print(response.choices[0].message.content)

Based on the provided context, the correct answer is **B. Because of a large number of vehicles, the area is polluted.**

**Reasoning from the text:** The passage states that in cities, "the environment is polluted with gases from vehicles," and this pollution results in a lack of fresh air, which directly causes urban residents to suffer from health issues like breathing problems.


## Next improvement

**After this, add:**
 1. PDF loader
 2. source citations
 3. metadata
 4. reranking
 5. conversation memory
 6. tool calling
 7. LangGraph workflow